In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sohansakib75/yuyvvty")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/yuyvvty


In [2]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("sohansakib75/yuyvvty")

print("Path to dataset files:", path)

# List subfolders and files inside the downloaded path
for root, dirs, files in os.walk(path):
    print(f"\nIn directory: {root}")
    for d in dirs:
        print("Subfolder:", d)
    for f in files:
        print("File:", f)


Path to dataset files: /kaggle/input/yuyvvty

In directory: /kaggle/input/yuyvvty
Subfolder: Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation

In directory: /kaggle/input/yuyvvty/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation
Subfolder: Eyelid
Subfolder: Normal
Subfolder: Cataract
Subfolder: Uveitis
Subfolder: Conjunctivitis

In directory: /kaggle/input/yuyvvty/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation/Eyelid
File: 266.jpeg
File: 228.jpeg
File: 761.jpeg
File: 833.jpeg
File: 130.jpeg
File: 158.jpeg
File: 977.jpeg
File: aug_990.jpg
File: 381.jpeg
File: 335.jpeg
File: aug_986.jpg
File: 265.jpeg
File: 843.jpeg
File: aug_962.jpg
File: aug_894.jpg
File: 251.jpeg
File: aug_896.jpg
File: 399.jpeg
File: 992.jpeg
File: 252.jpeg
File: 155.jpeg
File: 626.jpeg
File: 15

In [3]:
import cv2
import numpy as np
import os
from PIL import Image
from tqdm import tqdm

base_dir = "/kaggle/input/yuyvvty/Image Dataset on Eye Diseases Classification (Uveitis, Conjunctivitis, Cataract, Eyelid) with Symptoms and SMOTE Validation"
classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
TARGET_SIZE = (224, 224)

output_base = "/kaggle/working/preprocessed_all"

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def alpha_trimmed_mean_filter(img, kernel_size=3, alpha=2):
    pad = kernel_size // 2
    padded = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    filtered = np.zeros_like(img)
    for i in range(pad, padded.shape[0] - pad):
        for j in range(pad, padded.shape[1] - pad):
            kernel = padded[i - pad:i + pad + 1, j - pad:j + pad + 1].flatten()
            kernel.sort()
            trimmed = kernel[alpha:-alpha]
            filtered[i - pad, j - pad] = np.mean(trimmed)
    return filtered

def zoom_roi_with_padding(img, bbox, zoom_factor=1.3):
    h, w = img.shape[:2]
    x, y, bw, bh = bbox
    
    # Calculate ROI center
    cx = x + bw // 2
    cy = y + bh // 2
    
    # Calculate new ROI size
    new_w = int(bw * zoom_factor)
    new_h = int(bh * zoom_factor)
    
    # Calculate new bounding box with padding around ROI center
    x1 = max(cx - new_w // 2, 0)
    y1 = max(cy - new_h // 2, 0)
    x2 = min(cx + new_w // 2, w)
    y2 = min(cy + new_h // 2, h)
    
    # Crop and zoom ROI area
    roi = img[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, (w, h), interpolation=cv2.INTER_LINEAR)
    
    return roi_resized

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize full image first
    resized = cv2.resize(original_rgb, TARGET_SIZE)

    # Convert to grayscale for detection
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)

    # Detect eyes
    eyes = eye_cascade.detectMultiScale(gray, 1.3, 5)
    if len(eyes) > 0:
        bbox = eyes[0]
    else:
        # Detect face
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) > 0:
            bbox = faces[0]
        else:
            # If no detection, use whole image
            bbox = (0, 0, resized.shape[1], resized.shape[0])

    # Zoom ROI with padding, keep image size
    zoomed_img = zoom_roi_with_padding(resized, bbox, zoom_factor=1.3)

    # Apply CLAHE on grayscale
    gray_zoomed = cv2.cvtColor(zoomed_img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray_zoomed)

    # Apply alpha trimmed mean filter
    filtered_img = alpha_trimmed_mean_filter(clahe_img)

    # Convert back to RGB
    processed_img = cv2.merge([filtered_img]*3)

    return processed_img

# Process all images, save results
for cls in classes:
    class_input_dir = os.path.join(base_dir, cls)
    class_output_dir = os.path.join(output_base, cls)
    os.makedirs(class_output_dir, exist_ok=True)

    image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for i, img_file in enumerate(tqdm(image_files, desc=f"Processing {cls}")):
        img_path = os.path.join(class_input_dir, img_file)
        preprocessed_img = preprocess_image(img_path)
        if preprocessed_img is not None:
            save_path = os.path.join(class_output_dir, f"image_{i+1}.jpg")
            Image.fromarray(preprocessed_img).save(save_path)

print("All images preprocessed and saved successfully!")


Processing Conjunctivitis: 100%|██████████| 357/357 [02:15<00:00,  2.64it/s]

All images preprocessed and saved successfully!


In [4]:
import cv2
import numpy as np
import os
from PIL import Image
import random

preprocessed_dir = "/kaggle/working/preprocessed_all"  # Folder where all preprocessed images saved
augmented_dir = "/kaggle/working/augmented_all"

TARGET_SIZE = (224, 224)
classes_3aug = ["Cataract", "Conjunctivitis", "Eyelid", "Normal"]
classes_4aug = ["Uveitis"]

def random_brightness(img):
    factor = 0.7 + 0.6 * random.random()  # brightness factor between 0.7-1.3
    img = cv2.convertScaleAbs(img, alpha=factor, beta=0)
    return img

def random_rotation(img, angle_range=15):
    angle = random.uniform(-angle_range, angle_range)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return rotated

def horizontal_flip(img):
    return cv2.flip(img, 1)

def random_shift_zoom(img, shift_ratio=0.1, zoom_range=0.1):
    h, w = img.shape[:2]
    max_dx = int(w * shift_ratio)
    max_dy = int(h * shift_ratio)
    dx = random.randint(-max_dx, max_dx)
    dy = random.randint(-max_dy, max_dy)
    M_shift = np.float32([[1, 0, dx], [0, 1, dy]])
    shifted = cv2.warpAffine(img, M_shift, (w, h), borderMode=cv2.BORDER_REPLICATE)
    zx = 1 + random.uniform(-zoom_range, zoom_range)
    M_zoom = cv2.getRotationMatrix2D((w/2, h/2), 0, zx)
    zoomed = cv2.warpAffine(shifted, M_zoom, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return zoomed

def augment_image(img):
    aug_funcs = [random_brightness, random_rotation, horizontal_flip, random_shift_zoom]
    np.random.shuffle(aug_funcs)
    img_aug = img.copy()
    for func in aug_funcs[:2]:
        img_aug = func(img_aug)
    return img_aug

def save_augmented_images(class_name, image_paths, n_augments):
    save_dir = os.path.join(augmented_dir, class_name)
    os.makedirs(save_dir, exist_ok=True)

    for idx, orig_img_path in enumerate(image_paths):
        img = cv2.imread(orig_img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Save original image
        orig_save_path = os.path.join(save_dir, f"image_{idx}_0.jpg")
        Image.fromarray(img_rgb).save(orig_save_path)

        # Save augmented images
        for i in range(1, n_augments + 1):
            aug_img = augment_image(img_rgb)
            aug_save_path = os.path.join(save_dir, f"image_{idx}_{i}.jpg")
            Image.fromarray(aug_img).save(aug_save_path)

# Run for all classes
for cls in classes_3aug:
    class_dir = os.path.join(preprocessed_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    save_augmented_images(cls, image_files, 3)

for cls in classes_4aug:
    class_dir = os.path.join(preprocessed_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    save_augmented_images(cls, image_files, 6)  # Augment 6 times for Uveitis

print(f"All images augmented and saved to {augmented_dir}/")


All images augmented and saved to /kaggle/working/augmented_all/


In [9]:
import os
import random
from collections import defaultdict

augmented_dir = "/kaggle/working/augmented_all"
classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]

split_ratios = {'train': 0.8, 'val': 0.1, 'test': 0.1}

# Store split paths here
split_data = {'train': defaultdict(list), 'val': defaultdict(list), 'test': defaultdict(list)}

for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(image_files)

    n = len(image_files)
    n_train = int(split_ratios['train'] * n)
    n_val = int(split_ratios['val'] * n)
    n_test = n - n_train - n_val

    split_data['train'][cls] = image_files[:n_train]
    split_data['val'][cls] = image_files[n_train:n_train + n_val]
    split_data['test'][cls] = image_files[n_train + n_val:]

# Print summary table
print(f"{'Class':<15} | {'Train':<6} | {'Val':<6} | {'Test':<6} | {'Total':<6}")
print("-" * 50)
for cls in classes:
    n_train = len(split_data['train'][cls])
    n_val = len(split_data['val'][cls])
    n_test = len(split_data['test'][cls])
    total = n_train + n_val + n_test
    print(f"{cls:<15} | {n_train:<6} | {n_val:<6} | {n_test:<6} | {total:<6}")


Class           | Train  | Val    | Test   | Total 
--------------------------------------------------
Eyelid          | 1680   | 210    | 210    | 2100  
Normal          | 2076   | 259    | 261    | 2596  
Cataract        | 1740   | 217    | 219    | 2176  
Uveitis         | 1248   | 156    | 157    | 1561  
Conjunctivitis  | 1142   | 142    | 144    | 1428  


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import vgg19, VGG19_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, precision_recall_curve, auc
import time
import numpy as np
import os
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

augmented_dir = "/kaggle/working/augmented_all"

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

class EyeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

all_data = []
for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files = sorted(image_files)
    all_data.extend([(img_path, class_to_idx[cls]) for img_path in image_files])

random.shuffle(all_data)

k = 5
fold_size = len(all_data) // k
folds = []
for i in range(k):
    start_idx = i * fold_size
    if i == k - 1:
        fold = all_data[start_idx:]
    else:
        fold = all_data[start_idx:start_idx + fold_size]
    folds.append(fold)

def get_train_val_test(folds, test_fold_index, val_ratio=0.1):
    test_data = folds[test_fold_index]
    train_val_data = [item for i, f in enumerate(folds) if i != test_fold_index for item in f]
    n_val = int(len(train_val_data) * val_ratio)
    val_data = train_val_data[:n_val]
    train_data = train_val_data[n_val:]
    return train_data, val_data, test_data

def create_dataset_loader_from_list(data_list, transform, shuffle=False):
    image_paths = [item[0] for item in data_list]
    labels = [item[1] for item in data_list]
    dataset = EyeDataset(image_paths, labels, transform=transform)
    loader = DataLoader(dataset, batch_size=32, shuffle=shuffle, num_workers=2)
    return dataset, loader

criterion = nn.CrossEntropyLoss()

def train_epoch(model, dataloader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def eval_model(model, dataloader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds), np.array(all_probs)

num_epochs = 30
patience = 5
fold_test_accuracies = []

for fold_idx in range(k):
    print(f"\n===== Fold {fold_idx + 1} / {k} =====")
    train_data, val_data, test_data = get_train_val_test(folds, fold_idx)
    train_dataset, train_loader = create_dataset_loader_from_list(train_data, data_transforms['train'], shuffle=True)
    val_dataset, val_loader = create_dataset_loader_from_list(val_data, data_transforms['val'], shuffle=False)
    test_dataset, test_loader = create_dataset_loader_from_list(test_data, data_transforms['test'], shuffle=False)

    vgg19_model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
    vgg19_model.classifier = nn.Sequential(
        nn.Linear(25088, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(4096, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(4096, num_classes)
    )
    vgg19_model = vgg19_model.to(device)

    optimizer = optim.Adam(vgg19_model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_start_time = time.time()
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(vgg19_model, train_loader, optimizer)
        val_loss, val_acc, _, _, _ = eval_model(vgg19_model, val_loader)
        scheduler.step(val_loss)
        print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(vgg19_model.state_dict(), f"best_vgg19_fold{fold_idx}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    train_end_time = time.time()
    train_time = train_end_time - train_start_time
    print(f"Training time: {train_time:.2f} seconds")

    vgg19_model.load_state_dict(torch.load(f"best_vgg19_fold{fold_idx}.pth"))

    test_loss, test_acc, test_labels, test_preds, test_probs = eval_model(vgg19_model, test_loader)

    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')
    kappa = cohen_kappa_score(test_labels, test_preds)

    try:
        auc_macro = roc_auc_score(test_labels, test_probs, multi_class='ovo', average='macro')
    except Exception:
        auc_macro = float('nan')

    pr_auc_per_class = []
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(test_labels == i, test_probs[:, i])
        pr_auc_per_class.append(auc(recall, precision))
    pr_auc_macro = np.mean(pr_auc_per_class)

    print(f"Fold {fold_idx + 1} Test Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"AUC (macro): {auc_macro:.4f}")
    print(f"PR AUC (macro): {pr_auc_macro:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")

    fold_test_accuracies.append(acc)

print(f"\nAverage Test Accuracy over {k} folds: {np.mean(fold_test_accuracies):.4f} ± {np.std(fold_test_accuracies):.4f}")



===== Fold 1 / 5 =====
Epoch 1/30 | Train Loss: 0.6019 Acc: 0.7761 | Val Loss: 0.3399 Acc: 0.8617
Epoch 2/30 | Train Loss: 0.2481 Acc: 0.9141 | Val Loss: 0.2438 Acc: 0.9213
Epoch 3/30 | Train Loss: 0.1362 Acc: 0.9586 | Val Loss: 0.1251 Acc: 0.9581
Epoch 4/30 | Train Loss: 0.1267 Acc: 0.9594 | Val Loss: 0.2049 Acc: 0.9327
Epoch 5/30 | Train Loss: 0.1028 Acc: 0.9706 | Val Loss: 0.1156 Acc: 0.9581
Epoch 6/30 | Train Loss: 0.0433 Acc: 0.9863 | Val Loss: 0.1245 Acc: 0.9543
Epoch 7/30 | Train Loss: 0.0872 Acc: 0.9716 | Val Loss: 0.1117 Acc: 0.9695
Epoch 8/30 | Train Loss: 0.0519 Acc: 0.9835 | Val Loss: 0.1477 Acc: 0.9569
Epoch 9/30 | Train Loss: 0.0622 Acc: 0.9799 | Val Loss: 0.1867 Acc: 0.9492
Epoch 10/30 | Train Loss: 0.0468 Acc: 0.9842 | Val Loss: 0.1152 Acc: 0.9657
Epoch 11/30 | Train Loss: 0.0318 Acc: 0.9897 | Val Loss: 0.0373 Acc: 0.9886
Epoch 12/30 | Train Loss: 0.0005 Acc: 1.0000 | Val Loss: 0.0265 Acc: 0.9911
Epoch 13/30 | Train Loss: 0.0001 Acc: 1.0000 | Val Loss: 0.0250 Acc: 0.99

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
    precision_recall_curve, auc, classification_report, confusion_matrix, roc_curve
)
import time
import numpy as np
import os
import random
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

augmented_dir = "/kaggle/working/augmented_all"

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

class EyeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Prepare all data (image paths and labels)
all_data = []
for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files = sorted(image_files)
    all_data.extend([(img_path, class_to_idx[cls]) for img_path in image_files])

random.shuffle(all_data)

k = 5
fold_size = len(all_data) // k
folds = []
for i in range(k):
    start_idx = i * fold_size
    if i == k - 1:
        fold = all_data[start_idx:]
    else:
        fold = all_data[start_idx:start_idx + fold_size]
    folds.append(fold)

def get_train_val_test(folds, test_fold_index, val_ratio=0.1):
    test_data = folds[test_fold_index]
    train_val_data = [item for i, f in enumerate(folds) if i != test_fold_index for item in f]
    n_val = int(len(train_val_data) * val_ratio)
    val_data = train_val_data[:n_val]
    train_data = train_val_data[n_val:]
    return train_data, val_data, test_data

def create_dataset_loader_from_list(data_list, transform, shuffle=False):
    image_paths = [item[0] for item in data_list]
    labels = [item[1] for item in data_list]
    dataset = EyeDataset(image_paths, labels, transform=transform)
    loader = DataLoader(dataset, batch_size=32, shuffle=shuffle, num_workers=2)
    return dataset, loader

criterion = nn.CrossEntropyLoss()
num_epochs = 30
patience = 5

def train_epoch(model, dataloader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_model(model, dataloader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds), np.array(all_probs)

fold_test_accuracies = []

for fold_idx in range(k):
    print(f"\n===== Fold {fold_idx + 1} / {k} =====")
    train_data, val_data, test_data = get_train_val_test(folds, fold_idx)
    train_dataset, train_loader = create_dataset_loader_from_list(train_data, data_transforms['train'], shuffle=True)
    val_dataset, val_loader = create_dataset_loader_from_list(val_data, data_transforms['val'], shuffle=False)
    test_dataset, test_loader = create_dataset_loader_from_list(test_data, data_transforms['test'], shuffle=False)

    mobilenet_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    mobilenet_model.classifier[1] = nn.Linear(mobilenet_model.classifier[1].in_features, num_classes)
    mobilenet_model = mobilenet_model.to(device)

    optimizer = optim.Adam(mobilenet_model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_start_time = time.time()
    for epoch in range(num_epochs):
        start_epoch_time = time.time()
        train_loss, train_acc = train_epoch(mobilenet_model, train_loader, optimizer)
        val_loss, val_acc, _, _, _ = eval_model(mobilenet_model, val_loader)
        scheduler.step(val_loss)
        epoch_time = time.time() - start_epoch_time
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(mobilenet_model.state_dict(), f"best_mobilenetv2_fold{fold_idx}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    train_end_time = time.time()
    print(f"Training time fold {fold_idx+1}: {train_end_time - train_start_time:.2f} seconds")

    mobilenet_model.load_state_dict(torch.load(f"best_mobilenetv2_fold{fold_idx}.pth"))

    test_loss, test_acc, test_labels, test_preds, test_probs = eval_model(mobilenet_model, test_loader)

    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')
    kappa = cohen_kappa_score(test_labels, test_preds)

    try:
        auc_macro = roc_auc_score(test_labels, test_probs, multi_class='ovo', average='macro')
    except Exception:
        auc_macro = float('nan')

    pr_auc_per_class = []
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(test_labels == i, test_probs[:, i])
        pr_auc_per_class.append(auc(recall, precision))
    pr_auc_macro = np.mean(pr_auc_per_class)

    print(f"Fold {fold_idx + 1} Test Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"AUC (macro): {auc_macro:.4f}")
    print(f"PR AUC (macro): {pr_auc_macro:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")

    fold_test_accuracies.append(acc)

print(f"\nAverage Test Accuracy over {k} folds: {np.mean(fold_test_accuracies):.4f} ± {np.std(fold_test_accuracies):.4f}")



===== Fold 1 / 5 =====


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 80.6MB/s]


Epoch 1/30 | Train Loss: 0.4801 Acc: 0.8316 | Val Loss: 0.2282 Acc: 0.9150 | Time: 21.62s
Epoch 2/30 | Train Loss: 0.1204 Acc: 0.9610 | Val Loss: 0.1243 Acc: 0.9556 | Time: 21.65s
Epoch 3/30 | Train Loss: 0.0567 Acc: 0.9832 | Val Loss: 0.0717 Acc: 0.9721 | Time: 21.40s
Epoch 4/30 | Train Loss: 0.0326 Acc: 0.9906 | Val Loss: 0.1831 Acc: 0.9442 | Time: 21.46s
Epoch 5/30 | Train Loss: 0.0297 Acc: 0.9920 | Val Loss: 0.0426 Acc: 0.9860 | Time: 21.56s
Epoch 6/30 | Train Loss: 0.0143 Acc: 0.9969 | Val Loss: 0.0385 Acc: 0.9848 | Time: 21.48s
Epoch 7/30 | Train Loss: 0.0141 Acc: 0.9961 | Val Loss: 0.1006 Acc: 0.9721 | Time: 21.50s
Epoch 8/30 | Train Loss: 0.0270 Acc: 0.9904 | Val Loss: 0.0553 Acc: 0.9759 | Time: 21.55s
Epoch 9/30 | Train Loss: 0.0254 Acc: 0.9925 | Val Loss: 0.0493 Acc: 0.9835 | Time: 21.42s
Epoch 10/30 | Train Loss: 0.0105 Acc: 0.9970 | Val Loss: 0.0475 Acc: 0.9848 | Time: 21.53s
Epoch 11/30 | Train Loss: 0.0049 Acc: 0.9993 | Val Loss: 0.0319 Acc: 0.9873 | Time: 21.60s
Epoch 12

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
    precision_recall_curve, auc, classification_report, confusion_matrix, roc_curve
)
import time
import numpy as np
import os
import random
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

augmented_dir = "/kaggle/working/augmented_all"

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

class EyeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

all_data = []
for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files = sorted(image_files)
    all_data.extend([(img_path, class_to_idx[cls]) for img_path in image_files])

random.shuffle(all_data)

k = 5
fold_size = len(all_data) // k
folds = []
for i in range(k):
    start_idx = i * fold_size
    if i == k - 1:
        fold = all_data[start_idx:]
    else:
        fold = all_data[start_idx:start_idx + fold_size]
    folds.append(fold)

def get_train_val_test(folds, test_fold_index, val_ratio=0.1):
    test_data = folds[test_fold_index]
    train_val_data = [item for i, f in enumerate(folds) if i != test_fold_index for item in f]
    n_val = int(len(train_val_data) * val_ratio)
    val_data = train_val_data[:n_val]
    train_data = train_val_data[n_val:]
    return train_data, val_data, test_data

def create_dataset_loader_from_list(data_list, transform, shuffle=False):
    image_paths = [item[0] for item in data_list]
    labels = [item[1] for item in data_list]
    dataset = EyeDataset(image_paths, labels, transform=transform)
    loader = DataLoader(dataset, batch_size=32, shuffle=shuffle, num_workers=2)
    return dataset, loader

criterion = nn.CrossEntropyLoss()
num_epochs = 30
patience = 5

def train_epoch(model, dataloader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_model(model, dataloader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds), np.array(all_probs)

fold_test_accuracies = []

for fold_idx in range(k):
    print(f"\n===== Fold {fold_idx + 1} / {k} =====")
    train_data, val_data, test_data = get_train_val_test(folds, fold_idx)
    train_dataset, train_loader = create_dataset_loader_from_list(train_data, data_transforms['train'], shuffle=True)
    val_dataset, val_loader = create_dataset_loader_from_list(val_data, data_transforms['val'], shuffle=False)
    test_dataset, test_loader = create_dataset_loader_from_list(test_data, data_transforms['test'], shuffle=False)

    shufflenet_model = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
    shufflenet_model.fc = nn.Linear(shufflenet_model.fc.in_features, num_classes)
    shufflenet_model = shufflenet_model.to(device)

    optimizer = optim.Adam(shufflenet_model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_start_time = time.time()
    for epoch in range(num_epochs):
        start_epoch_time = time.time()
        train_loss, train_acc = train_epoch(shufflenet_model, train_loader, optimizer)
        val_loss, val_acc, _, _, _ = eval_model(shufflenet_model, val_loader)
        scheduler.step(val_loss)
        epoch_time = time.time() - start_epoch_time
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(shufflenet_model.state_dict(), f"best_shufflenetv2_fold{fold_idx}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    train_end_time = time.time()
    print(f"Training time fold {fold_idx+1}: {train_end_time - train_start_time:.2f} seconds")

    shufflenet_model.load_state_dict(torch.load(f"best_shufflenetv2_fold{fold_idx}.pth"))

    test_loss, test_acc, test_labels, test_preds, test_probs = eval_model(shufflenet_model, test_loader)

    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')
    kappa = cohen_kappa_score(test_labels, test_preds)

    try:
        auc_macro = roc_auc_score(test_labels, test_probs, multi_class='ovo', average='macro')
    except Exception:
        auc_macro = float('nan')

    pr_auc_per_class = []
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(test_labels == i, test_probs[:, i])
        pr_auc_per_class.append(auc(recall, precision))
    pr_auc_macro = np.mean(pr_auc_per_class)

    print(f"Fold {fold_idx + 1} Test Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"AUC (macro): {auc_macro:.4f}")
    print(f"PR AUC (macro): {pr_auc_macro:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")

    fold_test_accuracies.append(acc)

print(f"\nAverage Test Accuracy over {k} folds: {np.mean(fold_test_accuracies):.4f} ± {np.std(fold_test_accuracies):.4f}")



===== Fold 1 / 5 =====


Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth
100%|██████████| 8.79M/8.79M [00:00<00:00, 88.3MB/s]


Epoch 1/30 | Train Loss: 1.2586 Acc: 0.5758 | Val Loss: 0.8865 Acc: 0.7525 | Time: 12.87s
Epoch 2/30 | Train Loss: 0.6617 Acc: 0.8047 | Val Loss: 0.4351 Acc: 0.8921 | Time: 13.01s
Epoch 3/30 | Train Loss: 0.3540 Acc: 0.9062 | Val Loss: 0.2434 Acc: 0.9404 | Time: 13.15s
Epoch 4/30 | Train Loss: 0.2054 Acc: 0.9454 | Val Loss: 0.1518 Acc: 0.9581 | Time: 13.23s
Epoch 5/30 | Train Loss: 0.1196 Acc: 0.9717 | Val Loss: 0.0933 Acc: 0.9734 | Time: 13.25s
Epoch 6/30 | Train Loss: 0.0848 Acc: 0.9810 | Val Loss: 0.0917 Acc: 0.9708 | Time: 12.89s
Epoch 7/30 | Train Loss: 0.0503 Acc: 0.9897 | Val Loss: 0.0659 Acc: 0.9797 | Time: 13.16s
Epoch 8/30 | Train Loss: 0.0405 Acc: 0.9923 | Val Loss: 0.0638 Acc: 0.9797 | Time: 12.93s
Epoch 9/30 | Train Loss: 0.0346 Acc: 0.9930 | Val Loss: 0.0756 Acc: 0.9759 | Time: 12.71s
Epoch 10/30 | Train Loss: 0.0374 Acc: 0.9907 | Val Loss: 0.0635 Acc: 0.9848 | Time: 12.60s
Epoch 11/30 | Train Loss: 0.0234 Acc: 0.9955 | Val Loss: 0.0580 Acc: 0.9797 | Time: 13.41s
Epoch 12

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
    precision_recall_curve, auc, classification_report, confusion_matrix, roc_curve
)
import time
import numpy as np
import os
import random
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

augmented_dir = "/kaggle/working/augmented_all"  # Adjust path to your dataset root

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

class EyeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

all_data = []
for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files = sorted(image_files)
    all_data.extend([(img_path, class_to_idx[cls]) for img_path in image_files])

random.shuffle(all_data)

k = 5
fold_size = len(all_data) // k
folds = []
for i in range(k):
    start_idx = i * fold_size
    if i == k - 1:
        fold = all_data[start_idx:]
    else:
        fold = all_data[start_idx:start_idx + fold_size]
    folds.append(fold)

def get_train_val_test(folds, test_fold_index, val_ratio=0.1):
    test_data = folds[test_fold_index]
    train_val_data = [item for i, f in enumerate(folds) if i != test_fold_index for item in f]
    n_val = int(len(train_val_data) * val_ratio)
    val_data = train_val_data[:n_val]
    train_data = train_val_data[n_val:]
    return train_data, val_data, test_data

def create_dataset_loader_from_list(data_list, transform, shuffle=False):
    image_paths = [item[0] for item in data_list]
    labels = [item[1] for item in data_list]
    dataset = EyeDataset(image_paths, labels, transform=transform)
    loader = DataLoader(dataset, batch_size=32, shuffle=shuffle, num_workers=2)
    return dataset, loader

criterion = nn.CrossEntropyLoss()
num_epochs = 30
patience = 5

def train_epoch(model, dataloader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_model(model, dataloader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds), np.array(all_probs)

fold_test_accuracies = []

for fold_idx in range(k):
    print(f"\n===== Fold {fold_idx + 1} / {k} =====")
    train_data, val_data, test_data = get_train_val_test(folds, fold_idx)
    train_dataset, train_loader = create_dataset_loader_from_list(train_data, data_transforms['train'], shuffle=True)
    val_dataset, val_loader = create_dataset_loader_from_list(val_data, data_transforms['val'], shuffle=False)
    test_dataset, test_loader = create_dataset_loader_from_list(test_data, data_transforms['test'], shuffle=False)

    model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
    model.num_classes = num_classes
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_start_time = time.time()
    for epoch in range(num_epochs):
        start_epoch_time = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, optimizer)
        val_loss, val_acc, _, _, _ = eval_model(model, val_loader)
        scheduler.step(val_loss)
        epoch_time = time.time() - start_epoch_time
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"best_squeezenet_fold{fold_idx}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    train_end_time = time.time()
    print(f"Training time fold {fold_idx+1}: {train_end_time - train_start_time:.2f} seconds")

    model.load_state_dict(torch.load(f"best_squeezenet_fold{fold_idx}.pth"))

    test_loss, test_acc, test_labels, test_preds, test_probs = eval_model(model, test_loader)

    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')
    kappa = cohen_kappa_score(test_labels, test_preds)

    try:
        auc_macro = roc_auc_score(test_labels, test_probs, multi_class='ovo', average='macro')
    except Exception:
        auc_macro = float('nan')

    pr_auc_per_class = []
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(test_labels == i, test_probs[:, i])
        pr_auc_per_class.append(auc(recall, precision))
    pr_auc_macro = np.mean(pr_auc_per_class)

    print(f"Fold {fold_idx + 1} Test Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"AUC (macro): {auc_macro:.4f}")
    print(f"PR AUC (macro): {pr_auc_macro:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")

    fold_test_accuracies.append(acc)

print(f"\nAverage Test Accuracy over {k} folds: {np.mean(fold_test_accuracies):.4f} ± {np.std(fold_test_accuracies):.4f}")



===== Fold 1 / 5 =====


Downloading: "https://download.pytorch.org/models/squeezenet1_1-b8a52dc0.pth" to /root/.cache/torch/hub/checkpoints/squeezenet1_1-b8a52dc0.pth
100%|██████████| 4.73M/4.73M [00:00<00:00, 65.9MB/s]


Epoch 1/30 | Train Loss: 0.7720 Acc: 0.7068 | Val Loss: 0.4896 Acc: 0.8464 | Time: 11.70s
Epoch 2/30 | Train Loss: 0.4430 Acc: 0.8437 | Val Loss: 0.3305 Acc: 0.8731 | Time: 11.42s
Epoch 3/30 | Train Loss: 0.3321 Acc: 0.8809 | Val Loss: 0.3147 Acc: 0.8845 | Time: 11.36s
Epoch 4/30 | Train Loss: 0.2385 Acc: 0.9121 | Val Loss: 0.2677 Acc: 0.9061 | Time: 11.35s
Epoch 5/30 | Train Loss: 0.2073 Acc: 0.9259 | Val Loss: 0.2166 Acc: 0.9315 | Time: 11.15s
Epoch 6/30 | Train Loss: 0.1576 Acc: 0.9441 | Val Loss: 0.2440 Acc: 0.9074 | Time: 11.12s
Epoch 7/30 | Train Loss: 0.1209 Acc: 0.9590 | Val Loss: 0.1540 Acc: 0.9467 | Time: 11.20s
Epoch 8/30 | Train Loss: 0.0952 Acc: 0.9675 | Val Loss: 0.1109 Acc: 0.9581 | Time: 11.25s
Epoch 9/30 | Train Loss: 0.0921 Acc: 0.9689 | Val Loss: 0.1156 Acc: 0.9543 | Time: 11.08s
Epoch 10/30 | Train Loss: 0.0700 Acc: 0.9759 | Val Loss: 0.2349 Acc: 0.9213 | Time: 11.24s
Epoch 11/30 | Train Loss: 0.0539 Acc: 0.9823 | Val Loss: 0.0977 Acc: 0.9607 | Time: 11.22s
Epoch 12

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.models import shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, precision_recall_curve, auc
import time
import numpy as np
import os
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Eyelid", "Normal", "Cataract", "Uveitis", "Conjunctivitis"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

augmented_dir = "/kaggle/working/augmented_all"

data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class EyeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

all_data = []
for cls in classes:
    class_dir = os.path.join(augmented_dir, cls)
    image_files = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files = sorted(image_files)
    all_data.extend([(img_path, class_to_idx[cls]) for img_path in image_files])

random.shuffle(all_data)

k = 5
fold_size = len(all_data) // k
folds = []
for i in range(k):
    start_idx = i * fold_size
    if i == k - 1:
        fold = all_data[start_idx:]
    else:
        fold = all_data[start_idx:start_idx + fold_size]
    folds.append(fold)

def get_train_val_test(folds, test_fold_index, val_ratio=0.1):
    test_data = folds[test_fold_index]
    train_val_data = [item for i, f in enumerate(folds) if i != test_fold_index for item in f]
    n_val = int(len(train_val_data) * val_ratio)
    val_data = train_val_data[:n_val]
    train_data = train_val_data[n_val:]
    return train_data, val_data, test_data

def create_dataset_loader_from_list(data_list, transform, shuffle=False):
    image_paths = [item[0] for item in data_list]
    labels = [item[1] for item in data_list]
    dataset = EyeDataset(image_paths, labels, transform=transform)
    loader = DataLoader(dataset, batch_size=32, shuffle=shuffle, num_workers=2)
    return dataset, loader

# Feature extractor models

class ShuffleNetFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = nn.Sequential(*list(model.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.flatten(x)
        return x

class MobileNetFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.flatten(x)
        return x

# Fusion classifier

class FusionModel(nn.Module):
    def __init__(self, sn_dim, mn_dim, num_classes):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(sn_dim)
        self.bn2 = nn.BatchNorm1d(mn_dim)
        self.classifier = nn.Sequential(
            nn.Linear(sn_dim + mn_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        feat_sn, feat_mn = x
        feat_sn = self.bn1(feat_sn)
        feat_mn = self.bn2(feat_mn)
        fused = torch.cat((feat_sn, feat_mn), dim=1)
        out = self.classifier(fused)
        return out

criterion = nn.CrossEntropyLoss()

num_epochs = 30
patience = 5

fold_metrics = []

# Initialize base pretrained models once to avoid repeated download
shufflenet_base = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1).to(device)
mobilenet_base = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).to(device)

# Get feature dims from dummy input
with torch.no_grad():
    dummy = torch.randn(1,3,224,224).to(device)
    sn_feat_dim = ShuffleNetFeatureExtractor(shufflenet_base)(dummy).shape[1]
    mn_feat_dim = MobileNetFeatureExtractor(mobilenet_base)(dummy).shape[1]

print(f"ShuffleNet feature dim: {sn_feat_dim}, MobileNetV2 feature dim: {mn_feat_dim}")

for fold_idx in range(k):
    print(f"\n===== Fold {fold_idx + 1} / {k} =====")
    train_data, val_data, test_data = get_train_val_test(folds, fold_idx)
    train_dataset, train_loader = create_dataset_loader_from_list(train_data, data_transform, shuffle=True)
    val_dataset, val_loader = create_dataset_loader_from_list(val_data, data_transform, shuffle=False)
    test_dataset, test_loader = create_dataset_loader_from_list(test_data, data_transform, shuffle=False)

    # Create fresh models for each fold
    shufflenet_feat = ShuffleNetFeatureExtractor(shufflenet_base)
    mobilenet_feat = MobileNetFeatureExtractor(mobilenet_base)
    fusion_model = FusionModel(sn_feat_dim, mn_feat_dim, num_classes)

    shufflenet_feat = shufflenet_feat.to(device)
    mobilenet_feat = mobilenet_feat.to(device)
    fusion_model = fusion_model.to(device)

    params = list(fusion_model.parameters()) + list(shufflenet_feat.parameters()) + list(mobilenet_feat.parameters())
    optimizer = optim.Adam(params, lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    epochs_no_improve = 0

    # Training loop
    train_start_time = time.time()
    for epoch in range(num_epochs):
        shufflenet_feat.train()
        mobilenet_feat.train()
        fusion_model.train()

        running_loss = 0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            feat_sn = shufflenet_feat(inputs)
            feat_mn = mobilenet_feat(inputs)
            outputs = fusion_model((feat_sn, feat_mn))
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        shufflenet_feat.eval()
        mobilenet_feat.eval()
        fusion_model.eval()

        val_running_loss = 0
        val_correct = 0
        val_total = 0
        val_labels = []
        val_preds = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                feat_sn = shufflenet_feat(inputs)
                feat_mn = mobilenet_feat(inputs)
                outputs = fusion_model((feat_sn, feat_mn))
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                val_labels.extend(labels.cpu().numpy())
                val_preds.extend(preds.cpu().numpy())
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({
                'fusion': fusion_model.state_dict(),
                'sn_feat': shufflenet_feat.state_dict(),
                'mn_feat': mobilenet_feat.state_dict()
            }, f"best_fusion_fold{fold_idx}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    train_end_time = time.time()
    print(f"Training time: {train_end_time - train_start_time:.2f} seconds")

    # Load best weights
    checkpoint = torch.load(f"best_fusion_fold{fold_idx}.pth")
    fusion_model.load_state_dict(checkpoint['fusion'])
    shufflenet_feat.load_state_dict(checkpoint['sn_feat'])
    mobilenet_feat.load_state_dict(checkpoint['mn_feat'])

    # Test evaluation
    fusion_model.eval()
    shufflenet_feat.eval()
    mobilenet_feat.eval()

    test_running_loss = 0
    test_correct = 0
    test_total = 0
    test_labels = []
    test_preds = []
    test_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            feat_sn = shufflenet_feat(inputs)
            feat_mn = mobilenet_feat(inputs)
            outputs = fusion_model((feat_sn, feat_mn))
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)
            test_running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
            test_labels.extend(labels.cpu().numpy())
            test_preds.extend(preds.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())

    test_loss = test_running_loss / test_total
    test_acc = test_correct / test_total
    test_labels = np.array(test_labels)
    test_preds = np.array(test_preds)
    test_probs = np.array(test_probs)

    f1 = f1_score(test_labels, test_preds, average='macro')
    kappa = cohen_kappa_score(test_labels, test_preds)
    try:
        auc_macro = roc_auc_score(test_labels, test_probs, multi_class='ovo', average='macro')
    except Exception:
        auc_macro = float('nan')

    pr_auc_per_class = []
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(test_labels == i, test_probs[:, i])
        pr_auc_per_class.append(auc(recall, precision))
    pr_auc_macro = np.mean(pr_auc_per_class)

    print(f"Fold {fold_idx + 1} Test Metrics:")
    print(f"Accuracy: {test_acc:.4f}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"AUC (macro): {auc_macro:.4f}")
    print(f"PR AUC (macro): {pr_auc_macro:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")

    fold_metrics.append({
        'accuracy': test_acc,
        'f1': f1,
        'auc': auc_macro,
        'pr_auc': pr_auc_macro,
        'kappa': kappa
    })

# Average CV results
accs = [m['accuracy'] for m in fold_metrics]
f1s = [m['f1'] for m in fold_metrics]
aucs = [m['auc'] for m in fold_metrics if not np.isnan(m['auc'])]
pr_aucs = [m['pr_auc'] for m in fold_metrics]
kappas = [m['kappa'] for m in fold_metrics]

print(f"\n===== Cross-Validation Results =====")
print(f"Average Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Average F1-score (macro): {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
if aucs:
    print(f"Average AUC (macro): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
else:
    print("Average AUC (macro): NaN (skipped)")
print(f"Average PR AUC (macro): {np.mean(pr_aucs):.4f} ± {np.std(pr_aucs):.4f}")
print(f"Average Cohen's Kappa: {np.mean(kappas):.4f} ± {np.std(kappas):.4f}")


ShuffleNet feature dim: 1024, MobileNetV2 feature dim: 1280

===== Fold 1 / 5 =====
Epoch 1/30 | Train Loss: 0.4725 Acc: 0.8288 | Val Loss: 0.1440 Acc: 0.9607
Epoch 2/30 | Train Loss: 0.1187 Acc: 0.9616 | Val Loss: 0.0853 Acc: 0.9683
Epoch 3/30 | Train Loss: 0.0529 Acc: 0.9823 | Val Loss: 0.0719 Acc: 0.9784
Epoch 4/30 | Train Loss: 0.0331 Acc: 0.9906 | Val Loss: 0.0375 Acc: 0.9873
Epoch 5/30 | Train Loss: 0.0299 Acc: 0.9904 | Val Loss: 0.0502 Acc: 0.9822
Epoch 6/30 | Train Loss: 0.0211 Acc: 0.9931 | Val Loss: 0.0232 Acc: 0.9924
Epoch 7/30 | Train Loss: 0.0168 Acc: 0.9948 | Val Loss: 0.0634 Acc: 0.9784
Epoch 8/30 | Train Loss: 0.0246 Acc: 0.9923 | Val Loss: 0.0375 Acc: 0.9911
Epoch 9/30 | Train Loss: 0.0213 Acc: 0.9921 | Val Loss: 0.0397 Acc: 0.9835
Epoch 10/30 | Train Loss: 0.0201 Acc: 0.9946 | Val Loss: 0.0517 Acc: 0.9848
Epoch 11/30 | Train Loss: 0.0079 Acc: 0.9975 | Val Loss: 0.0251 Acc: 0.9898
Early stopping triggered
Training time: 326.37 seconds
Fold 1 Test Metrics:
Accuracy: 0.9